#  Build agentic legal research applications Lab HorizonDB diagnostics

Run these optional checks after Notebook 1. They are read-only except for opening database
connections. The notebook reports managed model aliases, AI Pipeline state, sink integrity,
DiskANN readiness, and source/sink linkage.


In [ ]:
import os
from pprint import pprint

import psycopg
from dotenv import load_dotenv

load_dotenv(override=True)

DB_CONFIG = {
    "host": os.environ["AZURE_PG_HOST"],
    "dbname": os.environ["AZURE_PG_NAME"],
    "user": os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port": os.environ["AZURE_PG_PORT"],
    "sslmode": os.environ.get("AZURE_PG_SSLMODE", "require"),
}

SOURCE_TABLE = "public.cases"
CHUNK_TABLE = "public.case_opinion_chunks"
PIPELINE_NAME = "case_opinion_embedding_pipeline"
DISKANN_INDEX_NAME = "idx_case_opinion_chunks_diskann"

print(f"Host: {DB_CONFIG['host']}")
print(f"Database: {DB_CONFIG['dbname']}")


## 1. Connectivity, extensions, and managed aliases

Expected lab aliases are `lab-chat` and `lab-embedding`, registered in Notebook 1.


In [ ]:
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("SELECT current_database(), current_user, version();")
    database, user, version = cur.fetchone()
    print(f"Database: {database}\nUser: {user}\nServer: {version}")

    cur.execute("""
        SELECT extname, extversion
        FROM pg_extension
        WHERE extname IN ('azure_ai', 'vector', 'pg_diskann', 'pg_textsearch', 'age', 'pg_durable')
        ORDER BY extname;
    """)
    print("\nExtensions:")
    for row in cur.fetchall():
        print("  ", row)

    print("\nManaged model registry:")
    cur.execute("SELECT * FROM model_registry.model_list_all();")
    columns = [description.name for description in cur.description]
    rows = cur.fetchall()
    print("  columns:", columns)
    for row in rows:
        print("  ", row)

    flattened = " ".join(str(value) for row in rows for value in row)
    expected = {"lab-chat", "lab-embedding"}
    print("\nMissing managed aliases:", sorted(alias for alias in expected if alias not in flattened))


## 2. Pipeline definition and run state

`ai.status()` is the primary troubleshooting surface. Durable-instance history is printed only
when `pg_durable` is installed and `df.instances` is available.


In [ ]:
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    print("Pipelines:")
    cur.execute("SELECT * FROM ai.list_pipelines();")
    pipeline_columns = [description.name for description in cur.description]
    for row in cur.fetchall():
        pprint(dict(zip(pipeline_columns, row)))

    print(f"\nStatus for {PIPELINE_NAME}:")
    cur.execute("SELECT * FROM ai.status(%s);", (PIPELINE_NAME,))
    status_columns = [description.name for description in cur.description]
    for row in cur.fetchall():
        pprint(dict(zip(status_columns, row)))

    print("\nCompiled definition:")
    try:
        cur.execute("SELECT ai.explain(%s);", (PIPELINE_NAME,))
        for row in cur.fetchall():
            print(row[0])
    except Exception as exc:
        print(f"ai.explain unavailable: {exc}")

    print("\nRecent durable runs (optional):")
    try:
        cur.execute(
            """
            SELECT id, label, status, created_at
            FROM df.instances
            WHERE label = %s
            ORDER BY created_at DESC
            LIMIT 5;
            """,
            (f"ai-pipeline:{PIPELINE_NAME}",),
        )
        for row in cur.fetchall():
            print("  ", row)
    except Exception as exc:
        print(f"  unavailable: {str(exc).splitlines()[0]}")


## 3. Sink rows, embeddings, and linkage

Healthy output has nonempty chunks, one 1536-dimension embedding per chunk, no missing copied
filter fields, no source cases without chunks, and no orphaned `doc_id` values.


In [ ]:
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute(f"""
        SELECT
            count(*) AS chunk_rows,
            count(DISTINCT doc_id) AS represented_cases,
            count(embedding) AS embeddings,
            count(*) FILTER (
                WHERE chunk_text IS NULL OR btrim(chunk_text) = ''
            ) AS empty_chunks,
            count(*) FILTER (
                WHERE court_level IS NULL OR decision_date IS NULL
            ) AS missing_filter_metadata
        FROM {CHUNK_TABLE};
    """)
    print("Sink summary:", cur.fetchone())

    cur.execute(f"""
        SELECT vector_dims(embedding), count(*)
        FROM {CHUNK_TABLE}
        WHERE embedding IS NOT NULL
        GROUP BY vector_dims(embedding)
        ORDER BY vector_dims(embedding);
    """)
    print("Embedding dimensions:", cur.fetchall())

    cur.execute(f"""
        SELECT count(*)
        FROM {SOURCE_TABLE} c
        WHERE NOT EXISTS (
            SELECT 1 FROM {CHUNK_TABLE} k WHERE k.doc_id = c.id
        );
    """)
    print("Source cases missing pipeline output:", cur.fetchone()[0])

    cur.execute(f"""
        SELECT count(*)
        FROM {CHUNK_TABLE} k
        LEFT JOIN {SOURCE_TABLE} c ON c.id = k.doc_id
        WHERE c.id IS NULL;
    """)
    print("Orphaned sink rows:", cur.fetchone()[0])

    cur.execute(f"""
        SELECT count(*)
        FROM {CHUNK_TABLE} k
        JOIN {SOURCE_TABLE} c ON c.id = k.doc_id
        WHERE k.court_level IS DISTINCT FROM c.court_level
           OR k.decision_date IS DISTINCT FROM c.decision_date;
    """)
    print("Source/sink filter metadata mismatches:", cur.fetchone()[0])


## 4. DiskANN index readiness

The vector index should be attached to `public.case_opinion_chunks.embedding` and report both
`indisvalid=true` and `indisready=true`.


In [ ]:
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            index_class.relname AS index_name,
            table_class.relname AS table_name,
            index_state.indisvalid,
            index_state.indisready,
            pg_get_indexdef(index_state.indexrelid) AS definition
        FROM pg_index AS index_state
        JOIN pg_class AS index_class ON index_class.oid = index_state.indexrelid
        JOIN pg_class AS table_class ON table_class.oid = index_state.indrelid
        JOIN pg_namespace AS namespace ON namespace.oid = table_class.relnamespace
        WHERE namespace.nspname = 'public'
          AND table_class.relname = 'case_opinion_chunks'
        ORDER BY index_class.relname;
        """
    )
    indexes = cur.fetchall()
    for row in indexes:
        print(row)

    diskann = [row for row in indexes if row[0] == DISKANN_INDEX_NAME]
    if not diskann:
        print(f"\nMISSING: {DISKANN_INDEX_NAME}")
    else:
        _, _, valid, ready, definition = diskann[0]
        print(f"\nDiskANN valid={valid} ready={ready}")
        print(definition)
